### Week 6, Day 2

We're about to create and use our own MCP Server and MCP Client!

It's pretty simple, but it's not super-simple. The excitment around MCP is about how easy it is to share and use other MCP Servers - making our own does involve a bit of work.

Let's review some python code made mostly by a hard-working Engineering Team:

accounts.py

In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown

load_dotenv(override=True)

True

In [2]:
from accounts import Account

In [6]:
account = Account.get("Ed")
account

Account(name='ed', balance=9630.262, strategy='', holdings={'AMZN': 6}, transactions=[3 shares of AMZN at 77.154 each., 3 shares of AMZN at 46.092 each.], portfolio_value_time_series=[('2026-03-30 14:57:31', 10047.538), ('2026-03-30 14:57:38', 9942.262)])

In [5]:
account.buy_shares("AMZN", 3, "Because this bookstore website looks promising")

'Completed. Latest details:\n{"name": "ed", "balance": 9630.262, "strategy": "", "holdings": {"AMZN": 6}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 77.154, "timestamp": "2026-03-30 14:57:31", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 3, "price": 46.092, "timestamp": "2026-03-30 14:57:38", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-03-30 14:57:31", 10047.538], ["2026-03-30 14:57:38", 9942.262]], "total_portfolio_value": 9942.262, "total_profit_loss": -57.737999999999374}'

In [7]:
account.report()

'{"name": "ed", "balance": 9630.262, "strategy": "", "holdings": {"AMZN": 6}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 77.154, "timestamp": "2026-03-30 14:57:31", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 3, "price": 46.092, "timestamp": "2026-03-30 14:57:38", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-03-30 14:57:31", 10047.538], ["2026-03-30 14:57:38", 9942.262], ["2026-03-30 14:57:49", 9888.262]], "total_portfolio_value": 9888.262, "total_profit_loss": -111.73799999999937}'

In [8]:
account.list_transactions()

[{'symbol': 'AMZN',
  'quantity': 3,
  'price': 77.154,
  'timestamp': '2026-03-30 14:57:31',
  'rationale': 'Because this bookstore website looks promising'},
 {'symbol': 'AMZN',
  'quantity': 3,
  'price': 46.092,
  'timestamp': '2026-03-30 14:57:38',
  'rationale': 'Because this bookstore website looks promising'}]

### Now we write an MCP server and use it directly!

In [9]:
# Now let's use our accounts server as an MCP server

params = {"command": "uv", "args": ["run", "accounts_server.py"]}
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()


In [10]:
mcp_tools

[Tool(name='get_balance', title=None, description='Get the cash balance of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_balanceArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'number'}}, 'required': ['result'], 'title': 'get_balanceOutput', 'type': 'object'}, icons=None, annotations=None, meta=None),
 Tool(name='get_holdings', title=None, description='Get the holdings of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_holdingsArguments', 'type': 'object'}, outputSchema={'additionalProperties': {'type': 'integer'}, 'title': 'get_holdingsDictOutput', 'type': 'object'}, icons=None, annotations=None, meta=None),
 Tool(name='buy_shares', 

In [11]:
instructions = "You are able to manage an account for a client, and answer questions about the account."
request = "My name is Ed and my account is under the name Ed. What's my balance and my holdings?"
model = "gpt-4.1-mini"

In [12]:

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="account_manager", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("account_manager"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))


Ed, your current cash balance is $9,630.26. In your holdings, you have 6 shares of Amazon (AMZN). If you need any further details or want to make any transactions, just let me know!

### Now let's build our own MCP Client

In [13]:
from accounts_client import get_accounts_tools_openai, read_accounts_resource, list_accounts_tools

mcp_tools = await list_accounts_tools()
print(mcp_tools)
openai_tools = await get_accounts_tools_openai()
print(openai_tools)

[Tool(name='get_balance', title=None, description='Get the cash balance of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_balanceArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'number'}}, 'required': ['result'], 'title': 'get_balanceOutput', 'type': 'object'}, icons=None, annotations=None, meta=None), Tool(name='get_holdings', title=None, description='Get the holdings of the given account name.\n\n    Args:\n        name: The name of the account holder\n    ', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_holdingsArguments', 'type': 'object'}, outputSchema={'additionalProperties': {'type': 'integer'}, 'title': 'get_holdingsDictOutput', 'type': 'object'}, icons=None, annotations=None, meta=None), Tool(name='buy_shares', ti

In [14]:
request = "My name is Ed and my account is under the name Ed. What's my balance?"

with trace("account_mcp_client"):
    agent = Agent(name="account_manager", instructions=instructions, model=model, tools=openai_tools)
    result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

Hi Ed, the current balance in your account is approximately $9,630.26. Is there anything specific you would like to do with your account today?

In [15]:
context = await read_accounts_resource("ed")
print(context)

{"name": "ed", "balance": 9630.262, "strategy": "", "holdings": {"AMZN": 6}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 77.154, "timestamp": "2026-03-30 14:57:31", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 3, "price": 46.092, "timestamp": "2026-03-30 14:57:38", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-03-30 14:57:31", 10047.538], ["2026-03-30 14:57:38", 9942.262], ["2026-03-30 14:57:49", 9888.262], ["2026-03-30 15:25:52", 9666.262]], "total_portfolio_value": 9666.262, "total_profit_loss": -333.7379999999994}


In [16]:
from accounts import Account
Account.get("ed").report()

'{"name": "ed", "balance": 9630.262, "strategy": "", "holdings": {"AMZN": 6}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 77.154, "timestamp": "2026-03-30 14:57:31", "rationale": "Because this bookstore website looks promising"}, {"symbol": "AMZN", "quantity": 3, "price": 46.092, "timestamp": "2026-03-30 14:57:38", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2026-03-30 14:57:31", 10047.538], ["2026-03-30 14:57:38", 9942.262], ["2026-03-30 14:57:49", 9888.262], ["2026-03-30 15:25:52", 9666.262], ["2026-03-30 15:25:54", 9984.262]], "total_portfolio_value": 9984.262, "total_profit_loss": -15.737999999999374}'

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercises</h2>
            <span style="color:#ff7800;">Make your own MCP Server! Make a simple function to return the current Date, and expose it as a tool so that an Agent can tell you today's date.<br/>Harder optional exercise: then make an MCP Client, and use a native OpenAI call (without the Agents SDK) to use your tool via your client.
            </span>
        </td>
    </tr>
</table>